Install and Update all required packages

In [ ]:
!pip uninstall -y transformers
!pip install -q --upgrade git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes
!pip install -U transformers accelerate bitsandbytes

Retrieve the model and set all associated settings

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# Config to make it run more efficient
bnb_config = BitsAndBytesConfig(
    load_in_4Bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

processor = AutoProcessor.from_pretrained(model_id)

# Set up the model
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    offload_buffers=True
)

Function to see if the model outputed the right answer

In [ ]:
import re

def extract_answer(text):
    # Look for total
    match = re.search(r"total[:\s]*(\d+)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # Look for the final answer
    match = re.search(r"final answer[:\s]*([^\n]+)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # Otherwise look for the last numbers returned
    numbers = re.findall(r"-?\d+\.?\d*", text)
    if numbers:
        return numbers[-1]

    return None

Run regular prompt

In [ ]:
from PIL import Image
import requests
from transformers import TextStreamer


streamer = TextStreamer(processor.tokenizer, skip_special_tokens=True)

# Image passed
image = Image.open("/content/Capture.PNG")

# Message passed to the model
messages = [
    {
        "role":"user",
        "content": [
            {"type": "image"},
            {"type": "text", "text":'''Count the total number of blocks in the pile using layer-by-layer reasoning.

                            For each layer (top to bottom), briefly note its structure and count both visible and logically hidden blocks. Keep descriptions minimal but sufficient to justify counts.

                            Output format:
                            Layer N: description — count
                            …
                            Total: X

                            Be concise. No extra explanation.'''}
        ]
    }
]

# Apply template to the input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(images=image, text=text, return_tensors="pt").to(model.device)

correct_count = 0
first_results = []
EXPECTED_ANSWER = "5"

# Run the input 50 times
for i in range(50):
    print(f"Starting iteration {i}...", flush=True)

    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1
    )

    decoded_text = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)  # ← renamed
    answer = extract_answer(decoded_text)  # ← uses renamed variable

    first_results.append(answer)

    if answer == EXPECTED_ANSWER:
        correct_count += 1

    print(f"Answer: {answer} | Correct: {correct_count}/{i + 1}", flush=True)

print(f"Final: {correct_count}/50")

Run loose prompt

In [ ]:
streamer = TextStreamer(processor.tokenizer, skip_special_tokens=True)

# Image to pass
image = Image.open("/content/Capture.PNG")

# Message passed to the model
messages = [
    {
        "role":"user",
        "content": [
            {"type": "image"},
            {"type": "text", "text":'''Look at this pile of blocks and count how many there are in total.

Think it through however makes sense to you — you might want to go layer by layer,
or count by columns, or whatever approach feels natural.

Just make sure your response ends with:
Total: X

where X is your final count.'''}
        ]
    }
]

# Apply template to the input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(images=image, text=text, return_tensors="pt").to(model.device)

correct_count = 0
second_results = []
EXPECTED_ANSWER = "5"

# Run the input 50 times
for i in range(50):
    print(f"Starting iteration {i}...", flush=True)

    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1
    )

    decoded_text = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)  # ← renamed
    answer = extract_answer(decoded_text)  # ← uses renamed variable

    second_results.append(answer)

    if answer == EXPECTED_ANSWER:
        correct_count += 1

    print(f"Answer: {answer} | Correct: {correct_count}/{i + 1}", flush=True)

print(f"Final: {correct_count}/50")

Run strict prompt

In [ ]:
streamer = TextStreamer(processor.tokenizer, skip_special_tokens=True)

# Image to pass
image = Image.open("/content/Capture.PNG")

# Message passed to the model
messages = [
    {
        "role":"user",
        "content": [
            {"type": "image"},
            {"type": "text", "text":'''You are a precise block-counting system. Follow these instructions exactly.

Analyze the image layer by layer from top to bottom.
For each layer you MUST output exactly this format:
Layer N: [shape of layer] — [count] blocks

Rules:
- Count ALL blocks including those hidden or inferred beneath visible ones
- Do not skip layers
- Do not add commentary or explanation
- Do not deviate from the output format

After all layers, output exactly:
Total: [number]

Any response not following this format exactly is incorrect.'''}
        ]
    }
]

# Apply template to the input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(images=image, text=text, return_tensors="pt").to(model.device)

correct_count = 0
third_results = []
EXPECTED_ANSWER = "5"

# Run the input 50 times
for i in range(50):
    print(f"Starting iteration {i}...", flush=True)

    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1
    )

    decoded_text = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)  # ← renamed
    answer = extract_answer(decoded_text)  # ← uses renamed variable

    third_results.append(answer)

    if answer == EXPECTED_ANSWER:
        correct_count += 1

    print(f"Answer: {answer} | Correct: {correct_count}/{i + 1}", flush=True)

print(f"Final: {correct_count}/50")